In [15]:
import wbgapi as wb
from tqdm import tqdm
import os

indicators = {
    'NY.GDP.MKTP.CD': 'gdp',
    'NY.GDP.PCAP.CD': 'gdp_per_capita',
    'FP.CPI.TOTL.ZG': 'inflation',
    'SL.UEM.TOTL.ZS': 'unemployment',
    'SP.URB.TOTL.IN.ZS': 'urban_share',
    'SE.TER.ENRR': 'school_tertiary',
    'SP.DYN.LE00.IN': 'life_expectancy',
    'SP.POP.TOTL': 'population',
    'SP.DYN.TFRT.IN': 'fertility_rate',
    'SH.DYN.MORT': 'under5_mortality',
    'SP.DYN.CDRT.IN': 'crude_death_rate',
    'SP.DYN.CBRT.IN': 'crude_birth_rate',
    'SM.POP.NETM': 'net_migration',
    'SP.POP.0014.TO.ZS': 'age_0_14_share',
    'SP.POP.65UP.TO.ZS': 'age_65_plus_share'
}

os.makedirs("data", exist_ok=True)

for code, name in tqdm(indicators.items(), desc="Downloading indicators"):
    df = wb.data.DataFrame(code, time=range(1960, 2024)).reset_index()
    df.to_csv(f"data/worldbank_{name}.csv", index=False)


In [18]:
import pandas as pd
import os

folder = "data"

files = [f for f in os.listdir(folder) if f.endswith(".csv")]

for f in files:
    path = os.path.join(folder, f)
    df = pd.read_csv(path)

    if  "country" not in df.columns:
        print(f"=== NO COUNTRY IN {f} ===")
        print(df.columns.tolist())
        print(df.head(1))


=== NO COUNTRY IN worldbank_age_0_14_share.csv ===
['economy', 'YR1960', 'YR1961', 'YR1962', 'YR1963', 'YR1964', 'YR1965', 'YR1966', 'YR1967', 'YR1968', 'YR1969', 'YR1970', 'YR1971', 'YR1972', 'YR1973', 'YR1974', 'YR1975', 'YR1976', 'YR1977', 'YR1978', 'YR1979', 'YR1980', 'YR1981', 'YR1982', 'YR1983', 'YR1984', 'YR1985', 'YR1986', 'YR1987', 'YR1988', 'YR1989', 'YR1990', 'YR1991', 'YR1992', 'YR1993', 'YR1994', 'YR1995', 'YR1996', 'YR1997', 'YR1998', 'YR1999', 'YR2000', 'YR2001', 'YR2002', 'YR2003', 'YR2004', 'YR2005', 'YR2006', 'YR2007', 'YR2008', 'YR2009', 'YR2010', 'YR2011', 'YR2012', 'YR2013', 'YR2014', 'YR2015', 'YR2016', 'YR2017', 'YR2018', 'YR2019', 'YR2020', 'YR2021', 'YR2022', 'YR2023']
  economy     YR1960     YR1961    YR1962     YR1963     YR1964     YR1965  \
0     ABW  42.512108  42.175482  41.86701  41.432243  40.846769  40.177006   

      YR1966     YR1967    YR1968  ...     YR2014     YR2015     YR2016  \
0  39.484108  38.962206  38.62738  ...  19.169508  19.034501  18.

In [ ]:
import pandas as pd
import os

folder = "data"
files = [f for f in os.listdir(folder) if f.endswith(".csv")]

long_tables = []

for f in files:
    name = f.replace("worldbank_", "").replace(".csv", "")
    path = os.path.join(folder, f)
    df = pd.read_csv(path)

    if "economy" not in df.columns:
        continue

    year_cols = [c for c in df.columns if c.startswith("YR")]
    df_long = df.melt(id_vars=["economy"], value_vars=year_cols, var_name="year", value_name=name)
    df_long["year"] = df_long["year"].str.replace("YR", "").astype(int)
    long_tables.append(df_long)

merged_wb_filtered = long_tables[0]
for t in long_tables[1:]:
    merged_wb_filtered = merged_wb_filtered.merge(t, on=["economy", "year"], how="outer")

merged_wb_filtered = merged_wb_filtered.sort_values(["economy", "year"]).reset_index(drop=True)
merged_wb_filtered = merged_wb_filtered[(merged_wb_filtered["year"] >= 1960) & (merged_wb_filtered["year"] <= 2023)]
merged_wb_filtered.to_csv("worldbank_merged_clean.csv", index=False)
print(merged_wb_filtered.columns)
print(merged_wb_filtered.head(1))


Index(['economy', 'year', 'age_0_14_share', 'age_65_plus_share',
       'crude_birth_rate', 'crude_death_rate', 'fertility_rate', 'gdp',
       'gdp_per_capita', 'inflation', 'life_expectancy', 'net_migration',
       'population', 'school_tertiary', 'under5_mortality', 'unemployment',
       'urban_share'],
      dtype='object')
  economy  year  age_0_14_share  age_65_plus_share  crude_birth_rate  \
0     ABW  1960       42.512108           2.855868            32.043   

   crude_death_rate  fertility_rate  gdp  gdp_per_capita  inflation  \
0             7.525           4.567  NaN             NaN        NaN   

   life_expectancy  net_migration  population  school_tertiary  \
0           64.049         -788.0     54922.0              NaN   

   under5_mortality  unemployment  urban_share  
0               NaN           NaN       50.776  


Есть NaN. Нужно проверять данные

In [24]:
import pandas as pd
import os

df = pd.read_csv("worldbank_merged_clean.csv")
cols_with_nan = [c for c in df.columns if df[c].isna().any() and c not in ("economy","year")]

for col in cols_with_nan:
    fname = f"worldbank_{col}.csv"
    path = os.path.join("data", fname)
    if os.path.exists(path):
        df_raw = pd.read_csv(path)
        print(f"=== {fname} ===")
        #print(df_raw.head(2))
        print()
    else:
        print(f"Missing file: {fname}")


=== worldbank_age_0_14_share.csv ===

=== worldbank_age_65_plus_share.csv ===

=== worldbank_crude_birth_rate.csv ===

=== worldbank_crude_death_rate.csv ===

=== worldbank_fertility_rate.csv ===

=== worldbank_gdp.csv ===

=== worldbank_gdp_per_capita.csv ===

=== worldbank_inflation.csv ===

=== worldbank_life_expectancy.csv ===

=== worldbank_net_migration.csv ===

=== worldbank_population.csv ===

=== worldbank_school_tertiary.csv ===

=== worldbank_under5_mortality.csv ===

=== worldbank_unemployment.csv ===

=== worldbank_urban_share.csv ===



In [ ]:
import pandas as pd
import glob
import os

pop_df = pd.read_csv("data/worldbank_population.csv")
small_pop_countries = pop_df[pop_df['YR2023'] < 1_000_000]['economy'].tolist()

files = sorted(glob.glob("data/worldbank_*.csv"))
ban_list_countries = set()
for path in files:
    df = pd.read_csv(path)
    
    if 'economy' in df.columns and any(c.startswith('YR') for c in df.columns):
        year_cols = [c for c in df.columns if c.startswith('YR')]
        df_long = df.melt(id_vars=['economy'], value_vars=year_cols, var_name='year', value_name='value')
        df_long['year'] = df_long['year'].str.replace('YR','').astype(int)
    else:
        continue

    df_long = df_long[~df_long['economy'].isin(small_pop_countries)]
    nan_countries = df_long[df_long['value'].isna()]['economy'].unique()
    
    print(os.path.basename(path), "countries with NaN:", len(nan_countries))
    if (len(nan_countries)<10):
        ban_list_countries.update(nan_countries)
    if ('gdp_per_capita' in fname):
        ban_list_countries.update(nan_countries)

worldbank_age_0_14_share.csv countries with NaN: 1
worldbank_age_65_plus_share.csv countries with NaN: 1
worldbank_crude_birth_rate.csv countries with NaN: 2
worldbank_crude_death_rate.csv countries with NaN: 2
worldbank_fertility_rate.csv countries with NaN: 2
worldbank_gdp.csv countries with NaN: 72
worldbank_gdp_per_capita.csv countries with NaN: 72
worldbank_inflation.csv countries with NaN: 183
worldbank_life_expectancy.csv countries with NaN: 3
worldbank_net_migration.csv countries with NaN: 1
worldbank_population.csv countries with NaN: 2
worldbank_school_tertiary.csv countries with NaN: 209
worldbank_under5_mortality.csv countries with NaN: 103
worldbank_unemployment.csv countries with NaN: 209
worldbank_urban_share.csv countries with NaN: 2


In [50]:
import pandas as pd
import glob
import os

files = sorted(glob.glob("data/worldbank_*.csv"))
small_pop_df = pd.read_csv("data/worldbank_population.csv")
small_pop_countries = small_pop_df[small_pop_df['YR2023'] < 1_000_000]['economy'].tolist()

selected_countries = set()

for path in files:
    fname = os.path.basename(path)
    df = pd.read_csv(path)
    if 'economy' in df.columns and any(c.startswith('YR') for c in df.columns):
        year_cols = [c for c in df.columns if c.startswith('YR')]
        df_long = df.melt(id_vars=['economy'], value_vars=year_cols, var_name='year', value_name='value')
        df_long['year'] = df_long['year'].str.replace('YR','').astype(int)
        df_long = df_long[~df_long['economy'].isin(small_pop_countries)]
        nan_countries = df_long[df_long['value'].isna()]['economy'].unique()
        if ('gdp' in fname) or ('gdp_per_capita' in fname) or (len(nan_countries) < 10):
            selected_countries.update(nan_countries)

print("Final list of countries to consider:")
print(sorted(selected_countries))



Final list of countries to consider:
['AFG', 'AGO', 'ALB', 'ARB', 'ARE', 'ARM', 'AZE', 'BGR', 'BHR', 'BIH', 'BLR', 'CEB', 'CUB', 'CYP', 'CZE', 'DJI', 'ECA', 'ERI', 'EST', 'ETH', 'GEO', 'GIN', 'GMB', 'GNB', 'GNQ', 'HRV', 'HUN', 'IDN', 'INX', 'ISR', 'JOR', 'KAZ', 'KGZ', 'KHM', 'KWT', 'LAO', 'LBN', 'LIC', 'LTU', 'LVA', 'MDA', 'MKD', 'MLI', 'MNG', 'MOZ', 'MRT', 'MWI', 'NAM', 'OSS', 'POL', 'PRK', 'PSE', 'PSS', 'QAT', 'ROU', 'RUS', 'SLV', 'SRB', 'SSD', 'SST', 'SVK', 'SVN', 'TEC', 'TJK', 'TKM', 'TLS', 'TUN', 'UKR', 'UZB', 'VEN', 'VNM', 'XKX', 'YEM']


Данные по России будем брать у Мэддисона(по России источник 1960-1990 ЦРУ). Проблема в том что данные из архивов СССР нет в электронной форме.

In [33]:
import pandas as pd

file_path = "maddison2023_web.dta"
df = pd.read_stata(file_path)

print(df.columns)
print(df.head(10))

Index(['countrycode', 'country', 'region', 'year', 'gdppc', 'pop'], dtype='object')
  countrycode      country                     region  year  gdppc  pop
0         AFG  Afghanistan  South and South East Asia     1    NaN  NaN
1         AFG  Afghanistan  South and South East Asia   730    NaN  NaN
2         AFG  Afghanistan  South and South East Asia  1000    NaN  NaN
3         AFG  Afghanistan  South and South East Asia  1090    NaN  NaN
4         AFG  Afghanistan  South and South East Asia  1150    NaN  NaN
5         AFG  Afghanistan  South and South East Asia  1252    NaN  NaN
6         AFG  Afghanistan  South and South East Asia  1253    NaN  NaN
7         AFG  Afghanistan  South and South East Asia  1254    NaN  NaN
8         AFG  Afghanistan  South and South East Asia  1255    NaN  NaN
9         AFG  Afghanistan  South and South East Asia  1256    NaN  NaN


In [40]:
import pandas as pd

file_path = "maddison2023_web.dta"
df = pd.read_stata(file_path)

df_filtered = df[
    (df['country'].isin(['Ukraine'])) &
    (df['year'].between(1960, 1990))
][['country', 'year', 'gdppc', 'pop']]

print(df_filtered)


        country  year    gdppc        pop
123321  Ukraine  1960      NaN  42644.035
123322  Ukraine  1961      NaN  43195.765
123323  Ukraine  1962      NaN  43697.245
123324  Ukraine  1963      NaN  44255.938
123325  Ukraine  1964      NaN  44785.626
123326  Ukraine  1965      NaN  45234.869
123327  Ukraine  1966      NaN  45673.640
123328  Ukraine  1967      NaN  46111.262
123329  Ukraine  1968      NaN  46510.301
123330  Ukraine  1969      NaN  46871.405
123331  Ukraine  1970      NaN  47235.697
123332  Ukraine  1971      NaN  47637.239
123333  Ukraine  1972      NaN  48026.627
123334  Ukraine  1973   7849.0  48367.002
123335  Ukraine  1974      NaN  48676.939
123336  Ukraine  1975      NaN  48973.428
123337  Ukraine  1976      NaN  49233.524
123338  Ukraine  1977      NaN  49453.675
123339  Ukraine  1978      NaN  49642.987
123340  Ukraine  1979      NaN  49835.012
123341  Ukraine  1980   8467.0  50046.649
123342  Ukraine  1981   8695.0  50235.677
123343  Ukraine  1982   8934.0  50

Данных по советским республикам скорее всего есть за 1980-1991

In [41]:
import pandas as pd

file_path = "maddison2023_web.dta"
df = pd.read_stata(file_path)

soviet_republics = [
    "Russian Federation", "Ukraine", "Belarus", "Estonia", "Latvia", "Lithuania",
    "Moldova", "Armenia", "Azerbaijan", "Georgia", "Kazakhstan", "Kyrgyzstan",
    "Tajikistan", "Turkmenistan", "Uzbekistan"
]

df_soviet = df[(df['country'].isin(soviet_republics)) & (df['year'].between(1960, 1991))]

nan_counts = df_soviet.groupby('country')['gdppc'].apply(lambda x: x.isna().sum())

print(nan_counts)


country
Armenia               19
Azerbaijan            19
Belarus               19
Estonia               19
Georgia               19
Kazakhstan            19
Kyrgyzstan            19
Latvia                19
Lithuania             19
Russian Federation     0
Tajikistan            19
Turkmenistan          19
Ukraine               19
Uzbekistan            19
Name: gdppc, dtype: int64


In [45]:
import pandas as pd

file_path = "maddison2023_web.dta"
df = pd.read_stata(file_path)

df_filtered = df[
    (df['country'].isin(['Russian Federation'])) &
    (df['year'].between(1960, 1990) | df['year'].between(2022, 2023))
][['country', 'year', 'gdppc', 'pop']]

print(df_filtered)


                   country  year         gdppc         pop
101593  Russian Federation  1960   5557.000000  119631.633
101594  Russian Federation  1961   5874.000000  121324.346
101595  Russian Federation  1962   6229.000000  122877.560
101596  Russian Federation  1963   6405.000000  124276.517
101597  Russian Federation  1964   6754.000000  125521.510
101598  Russian Federation  1965   7068.000000  126541.293
101599  Russian Federation  1966   7517.000000  127414.772
101600  Russian Federation  1967   7943.000000  128183.863
101601  Russian Federation  1968   8391.000000  128876.498
101602  Russian Federation  1969   8563.000000  129572.617
101603  Russian Federation  1970   9234.000000  130245.476
101604  Russian Federation  1971   9567.000000  130977.408
101605  Russian Federation  1972   9785.000000  131768.681
101606  Russian Federation  1973  10492.000000  132556.176
101607  Russian Federation  1974  10801.000000  133378.674
101608  Russian Federation  1975  11164.000000  134293.3

Будем использовать данные по gdppc РФ 1960-1991 Мэддисона. Уберём все страны у которых нет gpppc.

In [87]:
import pandas as pd
import glob
import os

pop_df = pd.read_csv("data/worldbank_population.csv")
small_pop_countries = pop_df[pop_df['YR2023'] < 1_000_000]['economy'].tolist()

files = sorted(glob.glob("data/worldbank_*.csv"))
ban_list_countries = set(small_pop_countries)
ban_list_metrics = set()
for path in files:
    df = pd.read_csv(path)
    fname = os.path.basename(path)
    if 'economy' in df.columns and any(c.startswith('YR') for c in df.columns):
        year_cols = [c for c in df.columns if c.startswith('YR')]
        df_long = df.melt(id_vars=['economy'], value_vars=year_cols, var_name='year', value_name='value')
        df_long['year'] = df_long['year'].str.replace('YR','').astype(int)
    else:
        continue

    df_long = df_long[~df_long['economy'].isin(small_pop_countries)]
    nan_countries = df_long[df_long['value'].isna()]['economy'].unique()
    
    print(os.path.basename(path), "countries with NaN:", len(nan_countries))
    if (len(nan_countries)<10 or 'gdp_per_capita' in fname):
        ban_list_countries.update(nan_countries)
    else:
        ban_list_metrics.add(fname)

ban_list_countries.discard('RUS')
print(len(ban_list_countries))
print(ban_list_metrics)

worldbank_age_0_14_share.csv countries with NaN: 1
worldbank_age_65_plus_share.csv countries with NaN: 1
worldbank_crude_birth_rate.csv countries with NaN: 2
worldbank_crude_death_rate.csv countries with NaN: 2
worldbank_fertility_rate.csv countries with NaN: 2
worldbank_gdp.csv countries with NaN: 72
worldbank_gdp_per_capita.csv countries with NaN: 72
worldbank_inflation.csv countries with NaN: 183
worldbank_life_expectancy.csv countries with NaN: 3
worldbank_net_migration.csv countries with NaN: 1
worldbank_population.csv countries with NaN: 2
worldbank_school_tertiary.csv countries with NaN: 209
worldbank_under5_mortality.csv countries with NaN: 103
worldbank_unemployment.csv countries with NaN: 209
worldbank_urban_share.csv countries with NaN: 2
129
{'worldbank_unemployment.csv', 'worldbank_inflation.csv', 'worldbank_school_tertiary.csv', 'worldbank_under5_mortality.csv', 'worldbank_gdp.csv'}


In [ ]:
import pandas as pd, glob, os
from functools import reduce

files = sorted(glob.glob("data/worldbank_*.csv"))

frames = []
for path in files:
    fname = os.path.basename(path)
    metric = fname.replace("worldbank_","").replace(".csv","")
    if fname in ban_list_metrics:
        continue
    df = pd.read_csv(path)
    if 'economy' in df.columns and any(c.startswith('YR') for c in df.columns):
        year_cols = [c for c in df.columns if c.startswith('YR')]
        df_long = df.melt(id_vars=['economy'], value_vars=year_cols,
                          var_name='year', value_name='value')
        df_long['year'] = df_long['year'].str.replace('YR','').astype(int)
        df_long = df_long[~df_long['economy'].isin(ban_list_countries)]
        df_long = df_long.rename(columns={'value': metric})
        frames.append(df_long[['economy','year',metric]])

merged_wb_filtered = reduce(lambda left,right: pd.merge(left,right,on=['economy','year'],how='outer'), frames)
merged_wb_filtered = merged_wb_filtered.sort_values(['economy','year']).reset_index(drop=True)
merged_wb_filtered.to_csv("final_dataset_filtered.csv", index=False)
print("Количество стран:", merged_wb_filtered['economy'].nunique())
print(merged_wb_filtered.head())


Количество стран: 137
  economy  year  age_0_14_share  age_65_plus_share  crude_birth_rate  \
0     AFE  1960       43.992245           2.985948         47.710772   
1     AFE  1961       44.098692           2.968138         47.731227   
2     AFE  1962       44.206120           2.951534         47.790919   
3     AFE  1963       44.332793           2.937356         47.831722   
4     AFE  1964       44.460765           2.926485         47.831555   

   crude_death_rate  fertility_rate  gdp_per_capita  life_expectancy  \
0         20.965499        6.650310      186.121835        44.169658   
1         20.745542        6.667308      186.941781        44.468838   
2         20.443071        6.688246      197.402402        44.877890   
3         20.234375        6.709226      225.440494        45.160583   
4         19.903705        6.724930      208.999748        45.535695   

   net_migration   population  urban_share  
0      -102704.0  130075728.0    14.577252  
1       -38646.0  1335

In [154]:
import pandas as pd

merged_wb_filtered = pd.read_csv("final_dataset_filtered.csv")

cols_to_drop = ['school_tertiary', 'under5_mortality', 'gdp']
merged_wb_filtered = merged_wb_filtered.drop(columns=[c for c in cols_to_drop if c in merged_wb_filtered.columns])

merged_wb_filtered.to_csv("final_dataset_filtered.csv", index=False)
print("Количество стран:", merged_wb_filtered['economy'].nunique())
print("Ban-list counties:", ban_list_countries)
merged_wb_filtered = merged_wb_filtered[~merged_wb_filtered['economy'].isin(ban_list_countries)]

merged_wb_filtered.to_csv("final_dataset_filtered.csv", index=False)
print("Количество стран после фильтрации бан-листа:", merged_wb_filtered['economy'].nunique())
print(merged_wb_filtered.head())

Количество стран: 137
Ban-list counties: {Ellipsis}
Количество стран после фильтрации бан-листа: 137
  economy  year  age_0_14_share  age_65_plus_share  crude_birth_rate  \
0     AFE  1960       43.992245           2.985948         47.710772   
1     AFE  1961       44.098692           2.968138         47.731227   
2     AFE  1962       44.206120           2.951534         47.790919   
3     AFE  1963       44.332793           2.937356         47.831722   
4     AFE  1964       44.460765           2.926485         47.831555   

   crude_death_rate  fertility_rate  gdp_per_capita  life_expectancy  \
0         20.965499        6.650310      186.121835        44.169658   
1         20.745542        6.667308      186.941781        44.468838   
2         20.443071        6.688246      197.402402        44.877890   
3         20.234375        6.709226      225.440494        45.160583   
4         19.903705        6.724930      208.999748        45.535695   

   net_migration   population  ur

In [ ]:
import pandas as pd

merged_wb_filtered = pd.read_csv("final_dataset_filtered.csv")

nan_countries = merged_wb_filtered[merged_wb_filtered.isna().any(axis=1)]['economy'].unique()
print("Страны с NaN:", nan_countries)
print("Количество стран с NaN:", len(nan_countries))
df_gdp_pc = pd.read_csv("data/worldbank_gdp_per_capita.csv")
abw_1960 = df_gdp_pc[df_gdp_pc['economy'] == 'ABW']['YR1960'].values
print("ABW GDP per capita 1960:", abw_1960)


Страны с NaN: ['RUS']
Количество стран с NaN: 1
ABW GDP per capita 1960: [nan]


Россия действительно 1 страна с отсутствующими данными, добавляем Мэддисона

In [ ]:
import pandas as pd

merged_wb_filtered = pd.read_csv("final_dataset_filtered.csv")
mask = (merged_wb_filtered['economy'] == 'RUS') & (merged_wb_filtered['year'].between(1990, 2020))
print(merged_wb_filtered.loc[mask, ['year', 'gdp_per_capita']])


# данные Мэддисона для России 1960-1990
maddison_df = pd.read_stata("data/maddison2023_web.dta")
maddison_rf = maddison_df[(maddison_df['country'] == 'Russian Federation') &
                          (maddison_df['year'].between(1960, 2000))][['year','gdppc']]

# подставляем только для годов 1960-1990
merged_wb_filtered.loc[mask, 'gdp_per_capita'] = merged_wb_filtered.loc[mask, 'year'].map(
    maddison_rf.set_index('year')['gdppc']
)
print(merged_wb_filtered.loc[mask, ['year', 'gdp_per_capita']])


      year  gdp_per_capita
6686  1990     3494.063232
6687  1991     3490.452393
6688  1992     3098.802734
6689  1993     2930.670166
6690  1994     2662.104004
6691  1995     2665.779785
6692  1996     2643.929199
6693  1997     2737.572021
6694  1998     1834.861816
6695  1999     1330.757202
6696  2000     1771.594116
6697  2001     2100.352539
6698  2002     2377.529541
6699  2003     2975.123047
6700  2004     4102.364746
6701  2005     5323.455078
6702  2006     6920.199707
6703  2007     9101.239258
6704  2008    11635.284180
6705  2009     8562.824219
6706  2010    10674.990234
6707  2011    14305.332031
6708  2012    15401.851562
6709  2013    15941.448242
6710  2014    14055.472656
6711  2015     9277.713867
6712  2016     8663.158203
6713  2017    10658.913086
6714  2018    11211.887695
6715  2019    11447.701172
6716  2020    10108.327148
      year  gdp_per_capita
6686  1990    12400.000000
6687  1991    12012.235521
6688  1992    10487.547701
6689  1993     9789.066759
6

In [ ]:
import pandas as pd

merged_wb_filtered = pd.read_csv("final_dataset_with_maddison.csv")

# только строки с NaN
nan_rows = merged_wb_filtered[merged_wb_filtered.isna().any(axis=1)]

for idx, row in nan_rows.iterrows():
    nan_cols = row[row.isna()].index.tolist()
    print(f"Строка index={idx}, страна={row['economy']}, NaN в колонках: {nan_cols}")


Строка index=6688, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6689, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6690, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6691, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6692, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6693, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6694, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6695, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6696, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6697, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6698, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6699, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6700, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6701, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка index=6702, страна=RUS, NaN в колонках: ['gdp_per_capita']
Строка ind

In [134]:
import pandas as pd

madd = pd.read_stata("data/maddison2023_web.dta")
sub = madd[(madd['year'] >= 1960) & (madd['year'] <= 1990)]
# сгруппировать по странам и посмотреть, у каких есть хотя бы один год в этом периоде
countries_with_data = sub['country'].unique()
print("Кол‑во стран с данными 1960‑1990:", len(countries_with_data))
print(countries_with_data)

Кол‑во стран с данными 1960‑1990: 169
['Afghanistan' 'Angola' 'Albania' 'United Arab Emirates' 'Argentina'
 'Armenia' 'Australia' 'Austria' 'Azerbaijan' 'Burundi' 'Belgium' 'Benin'
 'Burkina Faso' 'Bangladesh' 'Bulgaria' 'Bahrain' 'Bosnia and Herzegovina'
 'Belarus' 'Bolivia (Plurinational State of)' 'Brazil' 'Barbados'
 'Botswana' 'Central African Republic' 'Canada' 'Switzerland' 'Chile'
 'China' "Côte d'Ivoire" 'Cameroon' 'D.R. of the Congo' 'Congo' 'Colombia'
 'Comoros' 'Cabo Verde' 'Costa Rica' 'Czechoslovakia' 'Cuba' 'Cyprus'
 'Czech Republic' 'Germany' 'Djibouti' 'Dominica' 'Denmark'
 'Dominican Republic' 'Algeria' 'Ecuador' 'Egypt' 'Spain' 'Estonia'
 'Ethiopia' 'Finland' 'France' 'Gabon' 'United Kingdom' 'Georgia' 'Ghana'
 'Guinea' 'Gambia' 'Guinea-Bissau' 'Equatorial Guinea' 'Greece'
 'Guatemala' 'China, Hong Kong SAR' 'Honduras' 'Croatia' 'Haiti' 'Hungary'
 'Indonesia' 'India' 'Ireland' 'Iran (Islamic Republic of)' 'Iraq'
 'Iceland' 'Israel' 'Italy' 'Jamaica' 'Jordan' 'Japan' 

In [ ]:
import pandas as pd
import wbgapi as wb

# список стран с населением <1 млн (по твоей таблице)
pop = pd.read_csv("data/worldbank_population.csv")
small = set(pop[pop['YR2023'] < 1_000_000]['economy'])

# загружаем PPP GDP per capita (constant international $)
data = wb.data.DataFrame('NY.GDP.PCAP.PP.KD', time=range(1990, 2024)) \
         .reset_index() \
         .rename(columns={'economy':'iso3', 'value':'gdp_ppp_pc'})

data = data[~data['iso3'].isin(small)]
print(data.columns)

Index(['iso3', 'YR1990', 'YR1991', 'YR1992', 'YR1993', 'YR1994', 'YR1995',
       'YR1996', 'YR1997', 'YR1998', 'YR1999', 'YR2000', 'YR2001', 'YR2002',
       'YR2003', 'YR2004', 'YR2005', 'YR2006', 'YR2007', 'YR2008', 'YR2009',
       'YR2010', 'YR2011', 'YR2012', 'YR2013', 'YR2014', 'YR2015', 'YR2016',
       'YR2017', 'YR2018', 'YR2019', 'YR2020', 'YR2021', 'YR2022', 'YR2023'],
      dtype='object')


In [132]:

# приводим в длинный формат
year_cols = [c for c in data.columns if c.startswith('YR')]
data_long = data.melt(id_vars=['iso3'], value_vars=year_cols,
                      var_name='year', value_name='gdp_ppp_pc')
data_long['year'] = data_long['year'].str.replace('YR','').astype(int)

# фильтруем страны с полными данными (нет NaN за весь период)
good = data_long.groupby('iso3').filter(lambda df: df['gdp_ppp_pc'].notna().all())
countries_ok = set(good['iso3'])

print("Количество стран с полными PPP‑данными:", len(countries_ok))
print(sorted(list(countries_ok))[:20])

Количество стран с полными PPP‑данными: 196
['AFE', 'AFW', 'AGO', 'ALB', 'ARB', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BIH', 'BLR']


Нужно данные Мэддисона согласовать по коду страны с World Bank

In [135]:
maddison_to_iso3 = {
    'Afghanistan': 'AFG',
    'Angola': 'AGO',
    'Albania': 'ALB',
    'United Arab Emirates': 'ARE',
    'Argentina': 'ARG',
    'Armenia': 'ARM',
    'Australia': 'AUS',
    'Austria': 'AUT',
    'Azerbaijan': 'AZE',
    'Burundi': 'BDI',
    'Belgium': 'BEL',
    'Benin': 'BEN',
    'Burkina Faso': 'BFA',
    'Bangladesh': 'BGD',
    'Bulgaria': 'BGR',
    'Bahrain': 'BHR',
    'Bosnia and Herzegovina': 'BIH',
    'Belarus': 'BLR',
    'Bolivia (Plurinational State of)': 'BOL',
    'Brazil': 'BRA',
    'Barbados': 'BRB',
    'Botswana': 'BWA',
    'Central African Republic': 'CAF',
    'Canada': 'CAN',
    'Switzerland': 'CHE',
    'Chile': 'CHL',
    'China': 'CHN',
    "Côte d'Ivoire": 'CIV',
    'Cameroon': 'CMR',
    'D.R. of the Congo': 'COD',
    'Congo': 'COG',
    'Colombia': 'COL',
    'Comoros': 'COM',
    'Cabo Verde': 'CPV',
    'Costa Rica': 'CRI',
    'Czechoslovakia': None,  # исчезла, пропустить
    'Cuba': 'CUB',
    'Cyprus': 'CYP',
    'Czech Republic': 'CZE',
    'Germany': 'DEU',
    'Djibouti': 'DJI',
    'Dominica': 'DMA',
    'Denmark': 'DNK',
    'Dominican Republic': 'DOM',
    'Algeria': 'DZA',
    'Ecuador': 'ECU',
    'Egypt': 'EGY',
    'Spain': 'ESP',
    'Estonia': 'EST',
    'Ethiopia': 'ETH',
    'Finland': 'FIN',
    'France': 'FRA',
    'Gabon': 'GAB',
    'United Kingdom': 'GBR',
    'Georgia': 'GEO',
    'Ghana': 'GHA',
    'Guinea': 'GIN',
    'Gambia': 'GMB',
    'Guinea-Bissau': 'GNB',
    'Equatorial Guinea': 'GNQ',
    'Greece': 'GRC',
    'Guatemala': 'GTM',
    'China, Hong Kong SAR': 'HKG',
    'Honduras': 'HND',
    'Croatia': 'HRV',
    'Haiti': 'HTI',
    'Hungary': 'HUN',
    'Indonesia': 'IDN',
    'India': 'IND',
    'Ireland': 'IRL',
    'Iran (Islamic Republic of)': 'IRN',
    'Iraq': 'IRQ',
    'Iceland': 'ISL',
    'Israel': 'ISR',
    'Italy': 'ITA',
    'Jamaica': 'JAM',
    'Jordan': 'JOR',
    'Japan': 'JPN',
    'Kazakhstan': 'KAZ',
    'Kenya': 'KEN',
    'Kyrgyzstan': 'KGZ',
    'Cambodia': 'KHM',
    'Republic of Korea': 'KOR',
    'Kuwait': 'KWT',
    "Lao People's DR": 'LAO',
    'Lebanon': 'LBN',
    'Liberia': 'LBR',
    'Libya': 'LBY',
    'Saint Lucia': 'LCA',
    'Sri Lanka': 'LKA',
    'Lesotho': 'LSO',
    'Lithuania': 'LTU',
    'Luxembourg': 'LUX',
    'Latvia': 'LVA',
    'Morocco': 'MAR',
    'Republic of Moldova': 'MDA',
    'Madagascar': 'MDG',
    'Mexico': 'MEX',
    'TFYR of Macedonia': 'MKD',
    'Mali': 'MLI',
    'Malta': 'MLT',
    'Myanmar': 'MMR',
    'Montenegro': 'MNE',
    'Mongolia': 'MNG',
    'Mozambique': 'MOZ',
    'Mauritania': 'MRT',
    'Mauritius': 'MUS',
    'Malawi': 'MWI',
    'Malaysia': 'MYS',
    'Namibia': 'NAM',
    'Niger': 'NER',
    'Nigeria': 'NGA',
    'Nicaragua': 'NIC',
    'Netherlands': 'NLD',
    'Norway': 'NOR',
    'Nepal': 'NPL',
    'New Zealand': 'NZL',
    'Oman': 'OMN',
    'Pakistan': 'PAK',
    'Panama': 'PAN',
    'Peru': 'PER',
    'Philippines': 'PHL',
    'Poland': 'POL',
    'Puerto Rico': 'PRI',
    'D.P.R. of Korea': 'PRK',
    'Portugal': 'PRT',
    'Paraguay': 'PRY',
    'State of Palestine': 'PSE',
    'Qatar': 'QAT',
    'Romania': 'ROU',
    'Russian Federation': 'RUS',
    'Rwanda': 'RWA',
    'Saudi Arabia': 'SAU',
    'Sudan (Former)': 'SDN',
    'Senegal': 'SEN',
    'Singapore': 'SGP',
    'Sierra Leone': 'SLE',
    'El Salvador': 'SLV',
    'Serbia': 'SRB',
    'Sao Tome and Principe': 'STP',
    'Former USSR': None,
    'Slovakia': 'SVK',
    'Slovenia': 'SVN',
    'Sweden': 'SWE',
    'Swaziland': 'SWZ',
    'Seychelles': 'SYC',
    'Syrian Arab Republic': 'SYR',
    'Chad': 'TCD',
    'Togo': 'TGO',
    'Thailand': 'THA',
    'Tajikistan': 'TJK',
    'Turkmenistan': 'TKM',
    'Trinidad and Tobago': 'TTO',
    'Tunisia': 'TUN',
    'Turkey': 'TUR',
    'Taiwan, Province of China': 'TWN',
    'U.R. of Tanzania: Mainland': 'TZA',
    'Uganda': 'UGA',
    'Ukraine': 'UKR',
    'Uruguay': 'URY',
    'United States': 'USA',
    'Uzbekistan': 'UZB',
    'Venezuela (Bolivarian Republic of)': 'VEN',
    'Viet Nam': 'VNM',
    'Yemen': 'YEM',
    'South Africa': 'ZAF',
    'Zambia': 'ZMB',
    'Zimbabwe': 'ZWE'
}

maddison['iso3'] = maddison['country'].map(maddison_to_iso3)
maddison = maddison[maddison['iso3'].notna()]


In [140]:
import pandas as pd

# --- Maddison 1960-1990 ---
maddison = pd.read_stata("data/maddison2023_web.dta")
maddison = maddison[maddison['country'].isin(maddison_to_iso3.keys())]
maddison['iso3'] = maddison['country'].map(maddison_to_iso3)
maddison = maddison[maddison['iso3'].notna()]

# Приводим к длинному формату: year и gdppc
maddison_long = maddison[['iso3', 'year', 'gdppc']].copy()
maddison_long = maddison_long[maddison_long['year'].between(1960, 1990)]
maddison_long = maddison_long.rename(columns={'gdppc': 'gdp_ppp_pc'})

# --- WB PPP 1990+ ---
import wbgapi as wb

# исключаем мелкие страны
pop = pd.read_csv("data/worldbank_population.csv")
small = set(pop[pop['YR2023'] < 1_000_000]['economy'])

wb_data = wb.data.DataFrame('NY.GDP.PCAP.PP.KD', time=range(1991, 2024)).reset_index()
wb_data = wb_data.rename(columns={'economy':'iso3', 'value':'gdp_ppp_pc'})
wb_data = wb_data[~wb_data['iso3'].isin(small)]

# WB — в длинный формат
wb_long = wb_data.melt(id_vars=['iso3'], var_name='year', value_name='gdp_ppp_pc')
wb_long['year'] = wb_long['year'].str.replace('YR','').astype(int)

# --- Объединяем Maddison и WB ---
merged_ppp = pd.concat([maddison_long, wb_long], ignore_index=True)
merged_ppp = merged_ppp.sort_values(['iso3','year']).reset_index(drop=True)

# Проверим
print("Общее количество стран:", merged_ppp['iso3'].nunique())
print(merged_ppp.head(20))
print(merged_ppp.tail(20))



Общее количество стран: 221
   iso3  year   gdp_ppp_pc
0   AFE  1991  3292.714638
1   AFE  1992  3133.974586
2   AFE  1993  3022.859879
3   AFE  1994  2992.566301
4   AFE  1995  3047.018457
5   AFE  1996  3132.145150
6   AFE  1997  3176.422609
7   AFE  1998  3154.037654
8   AFE  1999  3156.349945
9   AFE  2000  3175.467232
10  AFE  2001  3207.570295
11  AFE  2002  3247.655911
12  AFE  2003  3263.224449
13  AFE  2004  3361.793730
14  AFE  2005  3482.940809
15  AFE  2006  3620.751947
16  AFE  2007  3766.897179
17  AFE  2008  3836.019073
18  AFE  2009  3768.186016
19  AFE  2010  3862.651279
      iso3  year   gdp_ppp_pc
12023  ZWE  2004  2828.843352
12024  ZWE  2005  2642.172286
12025  ZWE  2006  2519.827374
12026  ZWE  2007  2395.988843
12027  ZWE  2008  1949.034894
12028  ZWE  2009  2176.864511
12029  ZWE  2010  2563.478351
12030  ZWE  2011  2875.902061
12031  ZWE  2012  3301.166232
12032  ZWE  2013  3319.772086
12033  ZWE  2014  3352.381349
12034  ZWE  2015  3366.633713
12035  ZWE  201

In [143]:
import numpy as np

countries = merged_ppp['iso3'].drop_duplicates()
sample_countries = np.random.choice(countries, size=7, replace=False)

# фильтруем по выбранным странам и годам
subset = merged_ppp[(merged_ppp['iso3'].isin(sample_countries)) &
                    (merged_ppp['year'].isin([1990, 1991]))] \
            .sort_values(['iso3', 'year']) \
            .reset_index(drop=True)

print(subset)

   iso3  year    gdp_ppp_pc
0   ALB  1990   3983.000000
1   ALB  1991   4027.906145
2   CHL  1990  10203.000000
3   CHL  1991  11607.530403
4   CZE  1990  14178.000000
5   CZE  1991  24734.770206
6   IRL  1990  18838.000000
7   IRL  1991  29992.049285
8   IRN  1990   5620.000000
9   IRN  1991   9923.821238
10  LAO  1990   1481.000000
11  LAO  1991   2033.582583
12  SAS  1991   2207.152294


In [146]:
wb_data = wb.data.DataFrame('NY.GDP.PCAP.PP.KD', time=range(1960, 2024))
wb_data = wb_data.reset_index().rename(columns={'economy':'iso3', 'value':'gdp_ppp_pc'})

# Фильтруем маленькие страны
wb_ppp = wb_data[~wb_data['iso3'].isin(small_countries)].copy()

# Преобразуем в длинный формат
wb_ppp = wb_ppp.melt(id_vars=['iso3'], var_name='year', value_name='gdp_ppp_pc')
wb_ppp['year'] = wb_ppp['year'].str.replace('YR','').astype(int)

maddison_1990 = maddison[maddison['year'] == 1990]
wb_1990 = merged_ppp[merged_ppp['year'] == 1990]

# находим пересечение стран по iso3
common_countries_1990 = set(maddison_1990['iso3']).intersection(set(wb_1990['iso3']))
print("Количество стран с данными 1990 у обоих источников:", len(common_countries_1990))
print(sorted(list(common_countries_1990)))


Количество стран с данными 1990 у обоих источников: 166
['AFG', 'AGO', 'ALB', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BIH', 'BLR', 'BOL', 'BRA', 'BRB', 'BWA', 'CAF', 'CAN', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COL', 'COM', 'CPV', 'CRI', 'CUB', 'CYP', 'CZE', 'DEU', 'DJI', 'DMA', 'DNK', 'DOM', 'DZA', 'ECU', 'EGY', 'ESP', 'EST', 'ETH', 'FIN', 'FRA', 'GAB', 'GBR', 'GEO', 'GHA', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GTM', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LCA', 'LKA', 'LSO', 'LTU', 'LUX', 'LVA', 'MAR', 'MDA', 'MDG', 'MEX', 'MKD', 'MLI', 'MLT', 'MMR', 'MNE', 'MNG', 'MOZ', 'MRT', 'MUS', 'MWI', 'MYS', 'NAM', 'NER', 'NGA', 'NIC', 'NLD', 'NOR', 'NPL', 'NZL', 'OMN', 'PAK', 'PAN', 'PER', 'PHL', 'POL', 'PRI', 'PRK', 'PRT', 'PRY', 'PSE', 'QAT', 'ROU', 'RUS', 'RWA', 'SAU', 'SDN', 'SEN', 'SGP'

In [152]:
import pandas as pd
madd = pd.read_stata("data/maddison2023_web.dta")
r = madd[(madd['country']=="Russian Federation") & (madd['year']==2022)]
print(r)
nan_rows = madd[madd['gdppc'].isna()]
nan_countries = nan_rows['countrycode'].unique()
print("Страны с NaN в gdp_ppp_pc:", nan_countries)
print("Количество стран с NaN:", len(nan_countries))



       countrycode             country          region  year         gdppc  \
101655         RUS  Russian Federation  Eastern Europe  2022  25437.108022   

              pop  
101655  146692.81  
Страны с NaN в gdp_ppp_pc: ['AFG' 'AGO' 'ALB' 'ARE' 'ARG' 'ARM' 'AUS' 'AUT' 'AZE' 'BDI' 'BEL' 'BEN'
 'BFA' 'BGD' 'BGR' 'BHR' 'BIH' 'BLR' 'BOL' 'BRA' 'BRB' 'BWA' 'CAF' 'CAN'
 'CHE' 'CHL' 'CHN' 'CIV' 'CMR' 'COD' 'COG' 'COL' 'COM' 'CPV' 'CRI' 'CSK'
 'CUB' 'CYP' 'CZE' 'DEU' 'DJI' 'DMA' 'DNK' 'DOM' 'DZA' 'ECU' 'EGY' 'ESP'
 'EST' 'ETH' 'FIN' 'FRA' 'GAB' 'GBR' 'GEO' 'GHA' 'GIN' 'GMB' 'GNB' 'GNQ'
 'GRC' 'GTM' 'HKG' 'HND' 'HRV' 'HTI' 'HUN' 'IDN' 'IND' 'IRL' 'IRN' 'IRQ'
 'ISL' 'ISR' 'ITA' 'JAM' 'JOR' 'JPN' 'KAZ' 'KEN' 'KGZ' 'KHM' 'KOR' 'KWT'
 'LAO' 'LBN' 'LBR' 'LBY' 'LCA' 'LKA' 'LSO' 'LTU' 'LUX' 'LVA' 'MAR' 'MDA'
 'MDG' 'MEX' 'MKD' 'MLI' 'MLT' 'MMR' 'MNE' 'MNG' 'MOZ' 'MRT' 'MUS' 'MWI'
 'MYS' 'NAM' 'NER' 'NGA' 'NIC' 'NLD' 'NOR' 'NPL' 'NZL' 'OMN' 'PAK' 'PAN'
 'PER' 'PHL' 'POL' 'PRI' 'PRK' 'PRT' 'PRY' 'PS